# Marker diversity and morphology filter inspection

**Role:** Analysis.

Inspect source graphs and metadata to understand how sphericity and marker-diversity thresholds affect the available organoids. Compare retained/rejected examples and measured curvature before choosing training filters. Thresholds are exploratory settings, not model-derived conclusions. This notebook reads raw data and renders inspections; it does not fit a prediction model.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.


## Data and analysis settings

Select the dataset or saved run and the analysis options below. Model-dependent analyses restore the recorded inputs and fitted transformations before computing results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys
from pathlib import Path

PROJECT_ROOT = PROJECT_ROOT  # Locate the repository from the notebook’s working directory.
DATA_ROOT = PROJECT_ROOT / "training_data"  # Parent directory containing source datasets.

sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)

## Load the source cohort

Read graphs and auxiliary metadata from the selected dataset. No training split or model is required for this inspection.

In [ ]:
from src.data.io import load_graph_dataset_from_dir
from src.data.metadata import load_aux_metadata_for_dir, attach_metadata_to_graphs

# Load all organoids
# data_dir = PROJECT_ROOT / "training_data" / "smooth_cryptdist"
data_dir = PROJECT_ROOT / "training_data" / "mean_curvature_smooth"  # Dataset directory supplying graphs and metadata.

graphs = load_graph_dataset_from_dir(str(data_dir))
print(f"Loaded {len(graphs)} organoids.")
if graphs:
    print("Raw y shape:", tuple(graphs[0].y.shape))

meta = load_aux_metadata_for_dir(str(data_dir))
attach_metadata_to_graphs(graphs, meta)

In [ ]:
from src.data.io import select_graph_targets

# Gaussian curvature only, old 1D format
graphs = select_graph_targets(graphs, target_indices=0, inplace=False)

if graphs:
    print("Reduced y shape:", tuple(graphs[0].y.shape))


## Marker identities and metadata

Read the marker ordering and retain metadata needed to relate fate diversity to the original organoids.

In [ ]:
from src.data.metadata import snapshot_graph_metadata, load_marker_names_from_dir

marker_names = load_marker_names_from_dir(data_dir)
meta_lookup = snapshot_graph_metadata(graphs)

DEFAULT_MARKER_COLORS = {
    "Lysozyme": "#2C75D2",  # Display color for Lysozyme.
    "Serotonin": "#F392E3",  # Display color for Serotonin.
    "Mucin 2": "#3EC1D9",  # Display color for Mucin 2.
    "Chroma": "#D852CB",  # Display color for Chroma.
    "Glucagon": "#983EE2",  # Display color for Glucagon.
    "LGR5": "#FFB431",  # Display color for LGR5.
    "AldoB": "#F16C6A",  # Display color for AldoB.
    "Agr2": "#359BD5",  # Display color for Agr2.
    "KI67": "#808080",  # Display color for KI67.
    "none": "#EBEBEB",  # Display color for none.
}  # Display colors for cell identities; not model inputs.


## Inspect the sphericity filter

Separate retained and rejected organoids using the stated threshold, without changing the source dataset.

In [ ]:
from src.data.filters import filter_graphs_by_sphericity

graphs, rejected = filter_graphs_by_sphericity(
    graphs,
    max_sphericity=0.92,
    print_summary=True,
    return_rejected=True,
)

## Compare fate diversity

Calculate marker-diversity scores for both groups and inspect whether the morphology filter excludes compositionally complex examples.

In [ ]:
import matplotlib.pyplot as plt
from src.data.filters import marker_diversity_scores

# --- compute scores ---
scores_graphs = marker_diversity_scores(graphs)
scores_sphere = marker_diversity_scores(rejected)

# convert to numpy array if needed
scores_graphs = np.asarray(scores_graphs)
scores_sphere = np.asarray(scores_sphere)

# --- plot histogram ---
plt.figure(figsize=(6, 4))
plt.hist(scores_graphs, bins=30)
plt.xlabel("Marker diversity score")
plt.ylabel("Count")
plt.title("Distribution of marker diversity across organoids")
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(scores_sphere, bins=30)
plt.xlabel("Marker diversity score")
plt.ylabel("Count")
plt.title("Distribution of marker diversity across organoids")
plt.show()

## Render selected examples

Project marker identities onto selected organoid meshes to examine the filter decisions visually.

In [ ]:
from src.plotting.mesh_plots import project_marker_categories_for_graph, plot_categorical_mesh

N = 10  # Number of examples shown in the manual inspection.

top_idx = np.argsort(scores_sphere)[::-1][:N]

print("Top diversity spherical graphs:")
for rank, idx in enumerate(top_idx, start=1):
    g = rejected[idx]
    print(
        f"{rank:>2}. index={idx:<5} "
        f"score={scores_sphere[idx]:.4f} "
        f"organoid={getattr(g, 'organoid_str', None)}"
    )

for idx in top_idx:
    g = rejected[idx]

    marker_proj = project_marker_categories_for_graph(
        g,
        marker_names=marker_names,
        marker_colors=DEFAULT_MARKER_COLORS,
        meta_lookup=meta_lookup,
        missing_category="none",
    )

    fig = plot_categorical_mesh(
        marker_proj["mesh"],
        marker_proj["mesh_categories"],
        category_order=[m for m in DEFAULT_MARKER_COLORS.keys() if m != "none" and m in marker_names],
        category_colors=DEFAULT_MARKER_COLORS,
        baseline_color=DEFAULT_MARKER_COLORS.get("none", "lightgray"),
        baseline_name="none",
        add_legend=True,
        title=f"{getattr(g, 'organoid_str', None)} | diversity={scores_sphere[idx]:.4f}",
    )

    fig.show()